Projet ATRSC Partie 1:

Amitaï BERGER
Naïm EL OUASSAIDI

**Question 1:**

Si on regarde chaque type de produit indépendemment et qu'on calcule leur temps de service moyen sur l'ensemble des machines en sommant les temps d'attente moyens sur chaque machine du parcours du produit (et en inversant pour obtenir la capacité de service moyenne), on obtient les résultats suivants:

| Type | Intensité     | Capacité de service |
|-----------|-------------------------------|------------------------------------:|
| T1 |0.29 | 0.319 |
| T2 | 0.32 | 0.551 |
| T3 | 0.47 | 0.645 |
| T4 | 0.38 | 0.459 |

Donc pour tous les types, en moyenne, l'intensité est inférieure à la capacité.
De plus, on dispose d'au moins 4 employés pour chaque instance, en supposant qu'on puisse
avoir un employé sur chaque type de produit simultanément, on obtient bien qu'on a stabilité de la file, et donc un régime permanent existe








**Question 2:**

L'indicateur (i) va permettre d'avoir une idée du temps d'attente non productif du produit (en retirant le temps d'attente prodctif moyen du produit) et va donc très bien indiquer l'efficacité d'ALGO.

L'indicateyur (ii) devrait valoir la somme des intensités des différents types de produit (donc 1.46) s'il y a stabilité de la file. Il va donc seulement nous permettre d'évaluer l'écart à la situation de stabilité et sera certainement moins précis que l'indicateur (i) pour évaluer l'efficacité d'ALGO.

L'indicateur (iii) va nous permettre de calculer l'encombrement de la file grâce à la loi de Little mais reste moins précis que le (i)

**Question 3:**

Pour estimer une borne inférieure heuristique du temps de séjour moyen, on cherche à déterminer la durée minimale qu’un produit passerait dans l’atelier si tout fonctionnait de manière idéale.

Autrement dit, on suppose que :
* les machines sont toujours disponibles ;
* les employés sont immédiatement affectés ;
* aucun temps d’attente n’existe.

Dans cette situation parfaite, le temps de séjour minimal d’un produit de type Tₖ correspond simplement à la somme des temps de traitement moyens sur les machines de son parcours :

$$
E[Sk] = \sum{j \in \text{parcours}(Tk)} \frac{a{k,j} + b{k,j}}{2}
$$

Les temps de traitement moyens calculés pour chaque type de produit sont :

| Type | Parcours des machines     | Temps de traitement moyen total |
|-----------|-------------------------------|------------------------------------:|
| T1 | M1 → M2 → M3 → M4 → M8 | 2.64 |
| T2 | M2 → M4 → M7 | 1.16 |
| T3 | M3 → M5 → M1 | 1.10 |
| T4 | M5 → M6 → M7 → M8 | 2.72 |

En pondérant ces valeurs par les taux d’arrivée des produits  
(λ₁ = 0.29 ; λ₂ = 0.32 ; λ₃ = 0.47 ; λ₄ = 0.38), on obtient une estimation du temps de séjour moyen minimal pour l’ensemble de l’atelier :

$$
E[T{\text{séjour min}}]
= \frac{0.29×2.64 + 0.32×1.16 + 0.47×1.10 + 0.38×2.72}
{0.29 + 0.32 + 0.47 + 0.38}
≈ 1.8
$$

Cette valeur représente un plancher théorique :  
même avec une allocation optimale des employés et un fonctionnement sans aucune attente,  
le temps moyen passé par un produit dans le système ne pourrait pas être inférieur à environ 1.8 unités de temps.

En pratique, le temps observé sera plus élevé, en raison :
* des files d’attente sur certaines machines ;
* de la disponibilité limitée des employés qualifiés ;
* et de la variabilité aléatoire des arrivées et des durées de traitement.
`

**Question 4:**

Structure générale du simulateur

Le système est composé de trois entités principales : les produits, les machines et les employés.  
Chaque élément est représenté par une structure Julia spécifique.

* Product
  - id : identifiant unique du produit.  
  - typ : type de produit (1 à 4).  
  - arrival : instant d’arrivée dans l’atelier.  
  - route : liste ordonnée des machines à visiter.  
  - step : position actuelle dans le parcours.

* Machine
  - queue : file d’attente FIFO des produits en attente.  
  - inservice : produit actuellement en traitement (s’il y en a un).  
  - assignedemp : identifiant de l’employé affecté à la machine, ou nothing sinon.

* Employee
  - id : identifiant de l’employé.  
  - skills : liste des machines sur lesquelles il est qualifié à travailler.  
  - busy : indicateur booléen signalant si l’employé est occupé.  
  - waitingsince : instant où il est redevenu disponible.  
  - worktime : temps cumulé passé en activité depuis le début de la simulation.

* Workshop
  - Regroupe l’ensemble des objets précédents :
    - l’environnement SimJulia (env),
    - les listes de machines et d’employés,
    - la liste des durées de séjour complètes (completed),
    - ainsi qu’un compteur d’identifiants de produits (next_id).

Génération des événements
Arrivées de produits  
   Les produits de chaque type Tₖ arrivent dans l’atelier selon un processus de Poisson  
   d’intensité λₖ, donnée dans les paramètres.  
   Chaque produit suit ensuite le parcours prédéfini correspondant à son type (route).

Affectation aux machines  
   Lorsqu’un produit arrive à une machine :
   - s’il n’y a pas de produit en service, il entre directement dans la zone de travail ;
   - sinon, il est ajouté à la file d’attente FIFO associée à cette machine.

Attribution d’un employé  
   Si un produit est prêt sur une machine mais qu’aucun employé n’est encore affecté :  
   - le système cherche un employé disponible ayant la compétence sur cette machine ;
   - s’il y en a plusieurs, on choisit celui qui attend dans la salle d’attente depuis le plus longtemps ;
   - cet employé devient alors occupé et entame le traitement du produit.

Politique ALGO = FIFO

L’algorithme de décision FIFO (First In, First Out) est appliqué chaque fois qu’un employé termine une tâche :

L’employé examine l’ensemble des machines sur lesquelles il est qualifié.
Parmi ces machines, il identifie celles qui ont des produits en attente dans leur file.
Il choisit le produit qui est dans l’atelier depuis le plus longtemps, indépendamment de la machine où il se trouve.
Il se rend alors sur la machine correspondante et lance le traitement du produit.
Si aucune machine compatible n’a de produit en attente, l’employé retourne dans la salle d’attente et y reste jusqu’à ce qu’une nouvelle tâche devienne disponible.

Cette politique assure un traitement globalement équitable entre tous les produits : les plus anciens sont servis en priorité, à condition qu’un employé qualifié soit disponible.

Exécution d’une tâche de traitement

Lorsqu’un employé prend en charge un produit sur une machine :
* Le temps de traitement est tiré aléatoirement dans l’intervalle correspondant au type de produit et à la machine concernée.  
* Pendant ce temps, la machine est occupée, et aucun autre produit ne peut y entrer.
* À la fin du traitement :
  - le produit est retiré de la machine ;
  - la machine se libère et peut démarrer le traitement suivant (le prochain produit de sa file) ;
  - si le produit a encore d’autres machines dans son parcours, il se dirige vers la suivante ;
  - sinon, il quitte l’atelier, et sa durée totale de séjour est enregistrée.

Indicateurs mesurés

À la fin de la simulation, deux indicateurs principaux sont calculés :

Temps de séjour moyen  
   Moyenne des délais entre l’arrivée et la sortie des produits, représentative de la performance globale du système.

Taux d’occupation des employés  
   Rapport entre le temps passé à travailler et la durée totale de la simulation, pour chaque employé.  
   Cela permet d’évaluer la répartition de la charge de travail.

Objectif de la simulation

Cette simulation constitue la base expérimentale du projet.  
Elle permet :
* de vérifier la cohérence du modèle logique de l’atelier ;
* de mesurer la performance de la politique FIFO ;
* et de disposer de points de comparaison pour les futures optimisations dans la deuxième partie du projet (minimisation du temps de séjour moyen et équilibrage de charge).
``

In [30]:

using SimJulia
using Random
using Distributions
using ResumableFunctions
using Statistics
using Printf
using Plots

times = Float64[]
nb_pieces_per_times = Int[]
current_nb_of_pieces = 0

@resumable function piece(env::Environment, name::Int, machine::Resource)
    current_nb_of_pieces += 1
    println()
    push!(times, now(env))
    push!(nb_pieces, current_nbo[])

    @yield request(machine)
    service_time = rand(Exponential(1/8))
    @yield timeout(env, service_time)
    @yield release(machine)

    current_nbo[] -= 1
    push!(times, now(env))
    push!(nb_pieces, current_nbo[])
end

@resumable function arrival_process(env::Environment, machine::Resource, time_limit::Float64, current_nbo::Ref{Int})
    i = 0
    while now(env) < time_limit
        interarrival = rand(Exponential(1/3))
        if now(env) + interarrival > time_limit
            break
        end
        @yield timeout(env, interarrival)
        i += 1
        @process piece(env, i, machine, current_nbo)
    end
end

env = Simulation()
machine = Resource(env, 4)
@process arrival_process(env, machine, 100.0, current_nbo)
run(env)

function detect_steady_state(times, values; window=20, tol=0.05)
    n = length(values)
    for i in (2*window):n
        m1 = mean(values[i-window+1:i])
        m2 = mean(values[i-2*window+1:i-window])
        if abs(m1 - m2) < tol
            return times[i]
        end
    end
    return nothing
end

t_ss = detect_steady_state(times, nb_pieces)

if t_ss !== nothing
    indices = findall(t -> t >= t_ss, times)
    println("Nombre moyen de pièces (régime permanent) ≈ ", mean(nb_pieces[indices]))
end

plot(times, nb_pieces,
     xlabel="Temps",
     ylabel="Nombre de pièces",
     title="Évolution du système M/M/1",
     label="Nb pièces",
     lw=2)
if t_ss !== nothing
    vline!([t_ss], label="Régime permanent")
end

LoadError: ArgumentError: Package Distributions not found in current path.
- Run `import Pkg; Pkg.add("Distributions")` to install the Distributions package.

In [ ]:
import Pkg; Pkg.add("SimJulia")
using SimJulia, Random, Statistics

# Données
type_names = ["T1", "T2", "T3", "T4"]

arrival_rates = Dict(1=>0.29,2=>0.32,3=>0.47,4=>0.38)

routes = Dict(
    1=>[1,2,3,4,8],
    2=>[2,4,7],
    3=>[3,5,1],
    4=>[5,6,7,8]
)

service_times = Dict(
    (1,1)=>(0.58,0.78),(1,2)=>(0.23,0.56),(1,3)=>(0.81,0.93),(1,4)=>(0.12,0.39),(1,8)=>(0.82,1.04),
    (2,2)=>(0.59,0.68),(2,4)=>(0.74,0.77),(2,7)=>(0.30,0.55),
    (3,1)=>(0.57,0.64),(3,3)=>(0.37,0.54),(3,5)=>(0.35,0.63),
    (4,5)=>(0.36,0.51),(4,6)=>(0.61,0.70),(4,7)=>(0.78,0.85),(4,8)=>(0.18,0.37)
)

Q = [
    1 1 0 0 0 0 0 0;
    0 0 1 1 0 0 0 0;
    0 0 0 0 1 1 0 0;
    0 0 0 0 0 0 1 1
]

# Structures
mutable struct Product
    id::Int
    typ::Int
    arrival::Float64
    route::Vector{Int}
    step::Int
end

mutable struct Machine
    queue::Vector{Product}
    in_service::Union{Nothing,Product}
    assigned::Union{Nothing,Int}
end

mutable struct Employee
    id::Int
    skills::Vector{Int}
    busy::Bool
    wait_since::Float64
    work_time::Float64
end

mutable struct Workshop
    env::Environment
    machines::Vector{Machine}
    employees::Vector{Employee}
    done::Vector{Float64}
    nextid::Int
end

exp_time(rate) = -log(rand()) / rate

function serv_time(t,m)
    a,b = service_times[(t,m)]
    return rand()*(b-a) + a
end

function assign!(ws::Workshop, m::Int)
    mach = ws.machines[m]

    if mach.in_service === nothing || mach.assigned !== nothing
        return
    end

    free_emp = findfirst(e -> (!e.busy && (m in e.skills)), ws.employees)

    free_emp === nothing && return

    mach.assigned = free_emp
    emp = ws.employees[free_emp]
    emp.busy = true

    @process employee_process(ws, free_emp, m)
end

@resumable function employee_process(ws::Workshop, eid::Int, m::Int)
    env = ws.env
    emp = ws.employees[eid]
    mach = ws.machines[m]
    prod = mach.in_service

    serv = serv_time(prod.typ, m)

    start_time = now(env)
    @yield timeout(env, serv)

    emp.work_time += now(env) - start_time

    mach.in_service = nothing
    mach.assigned = nothing

    prod.step += 1

    if prod.step > length(prod.route)
        push!(ws.done, now(env) - prod.arrival)
    else
        nextm = prod.route[prod.step]
        product_arrive!(ws, prod, nextm)
    end

    emp.busy = false
    emp.wait_since = now(env)
end

function product_arrive!(ws::Workshop, p::Product, m::Int)
    mach = ws.machines[m]

    if mach.in_service === nothing
        mach.in_service = p
        assign!(ws, m)
    else
        push!(mach.queue, p)
    end
end

@resumable function arrivals(ws::Workshop, t::Int, limit::Float64)
    env = ws.env

    while now(env) < limit
        @yield timeout(env, exp_time(arrival_rates[t]))

        ws.nextid += 1
        p = Product(ws.nextid, t, now(env), routes[t], 1)

        product_arrive!(ws, p, p.route[1])
    end
end

function simulate(horizon=5000.0)
    env = Simulation()

    m = 8
    q = 4

    machines = [Machine(Product[], nothing, nothing) for _ in 1:m]

    employees = [
        Employee(i, [j for j in 1:m if Q[i,j] == 1], false, 0.0, 0.0)
        for i in 1:q
    ]

    ws = Workshop(env, machines, employees, Float64[], 0)

    for t in 1:4
        @process arrivals(ws, t, horizon)
    end

    run(env, horizon)

    return mean(ws.done),
           [e.work_time/horizon for e in ws.employees],
           length(ws.done)
end

meanT, loads, n = simulate(3000.0)

println("Temps de séjour moyen ≈ ", round(meanT, digits=3))
println("Occupations employés : ", round.(loads, digits=3))
println("Produits traités : ", n)

   Resolving package versions...
  No Changes to `C:\Users\amita\.julia\environments\v1.11\Project.toml`
  No Changes to `C:\Users\amita\.julia\environments\v1.11\Manifest.toml`
